# Populate Contoso Lakehouse

Creates 3 KPI tables in **contoso_lakehouse** for the Semantic Model.
- ProjectKPIs (4 rows — division-level project health)
- SafetyKPIs (48 rows — monthly safety metrics by division)
- FleetKPIs (4 rows — division-level fleet health)

**Run in:** Fabric portal → contoso_lakehouse → Open notebook → Run all

In [ ]:
import random, datetime
random.seed(42)

DIVISIONS = ["Division-Alpha", "Division-Beta", "Division-Gamma", "Division-Delta"]

DIVISION_PROFILES = {
    "Division-Alpha": {"project_count": 15, "cpi_mean": 0.95, "spi_mean": 0.92, "budget_total": 28.5e9, "fleet_count": 45, "fleet_op_pct": 0.88, "incident_base": 35, "maint_cost_base": 2500000},
    "Division-Beta": {"project_count": 12, "cpi_mean": 1.02, "spi_mean": 0.98, "budget_total": 14.4e9, "fleet_count": 60, "fleet_op_pct": 0.91, "incident_base": 28, "maint_cost_base": 3200000},
    "Division-Gamma": {"project_count": 10, "cpi_mean": 0.98, "spi_mean": 1.01, "budget_total": 4.5e9, "fleet_count": 25, "fleet_op_pct": 0.85, "incident_base": 18, "maint_cost_base": 1100000},
    "Division-Delta": {"project_count": 5, "cpi_mean": 1.05, "spi_mean": 1.03, "budget_total": 15.0e9, "fleet_count": 10, "fleet_op_pct": 0.95, "incident_base": 8, "maint_cost_base": 400000},
}

In [ ]:
# Generate ProjectKPIs
project_rows = []
for div in DIVISIONS:
    p = DIVISION_PROFILES[div]
    actual = p["budget_total"] * (1 / p["cpi_mean"])
    red_count = max(1, int(p["project_count"] * random.uniform(0.10, 0.20)))
    project_rows.append({
        "division": div, "total_budget": round(p["budget_total"], 0),
        "total_actual_cost": round(actual, 0), "avg_cpi": p["cpi_mean"],
        "avg_spi": p["spi_mean"], "project_count": p["project_count"],
        "red_projects": red_count,
    })

spark.createDataFrame(project_rows).write.mode("overwrite").format("delta").saveAsTable("contoso_lakehouse.projectkpis")
print(f"[OK] ProjectKPIs: {len(project_rows)} rows")

In [ ]:
# Generate SafetyKPIs
safety_rows = []
base_date = datetime.date(2025, 1, 1)
for month_offset in range(12):
    year = base_date.year + (base_date.month + month_offset - 1) // 12
    month_num = (base_date.month + month_offset - 1) % 12 + 1
    month_str = f"{year}-{month_num:02d}-01"
    for div in DIVISIONS:
        p = DIVISION_PROFILES[div]
        seasonal = 1.0 + 0.1 * (month_offset % 3 - 1)
        incidents = max(0, int(p["incident_base"] * seasonal * random.uniform(0.7, 1.3)))
        near_misses = int(incidents * 0.25)
        ltis = max(0, incidents - near_misses - int(incidents * random.uniform(0.3, 0.5)))
        lost_time = round(ltis * random.uniform(1.0, 5.0), 1)
        safety_rows.append({
            "division": div, "month": month_str, "total_incidents": incidents,
            "near_misses": near_misses, "ltis": ltis, "lost_time_days": lost_time,
        })

spark.createDataFrame(safety_rows).write.mode("overwrite").format("delta").saveAsTable("contoso_lakehouse.safetykpis")
print(f"[OK] SafetyKPIs: {len(safety_rows)} rows")

In [ ]:
# Generate FleetKPIs
fleet_rows = []
for div in DIVISIONS:
    p = DIVISION_PROFILES[div]
    operational = int(p["fleet_count"] * p["fleet_op_pct"])
    breakdowns = max(0, int(p["fleet_count"] * random.uniform(0.05, 0.15)))
    fleet_rows.append({
        "division": div, "total_equipment": p["fleet_count"],
        "operational_count": operational,
        "availability_pct": round(p["fleet_op_pct"] * 100, 1),
        "total_maintenance_cost": round(p["maint_cost_base"] * 12, 0),
        "breakdown_count": breakdowns,
    })

spark.createDataFrame(fleet_rows).write.mode("overwrite").format("delta").saveAsTable("contoso_lakehouse.fleetkpis")
print(f"[OK] FleetKPIs: {len(fleet_rows)} rows")
print("\n✅ All 3 tables written to contoso_lakehouse")

## Create Semantic Model (Direct Lake)

Uses Fabric **Semantic Link** (`sempy`) to create a Direct Lake semantic model
from the lakehouse tables above. Replaces manual model creation in the portal.

In [ ]:
import sempy.fabric as fabric

MODEL_NAME = "Contoso_KPI_Model"
LAKEHOUSE  = "contoso_lakehouse"

# Check if model already exists
existing = fabric.list_datasets()
if MODEL_NAME in existing['Dataset Name'].values:
    print(f"[SKIP] Semantic model '{MODEL_NAME}' already exists")
else:
    fabric.create_semantic_model_from_default_lakehouse(
        dataset=MODEL_NAME,
        lakehouse=LAKEHOUSE,
    )
    print(f"\u2705 Semantic model '{MODEL_NAME}' created (Direct Lake over {LAKEHOUSE})")